In [1]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

drive.mount("/content/drive")   # idempotent
CKPT_ROOT = Path(read_yaml(REPO_DIR / "configs" / "paths.yaml")["ckpt_root"])
print("Repo dir:", REPO_DIR)
print("Checkpoint root:", CKPT_ROOT)

Mounted at /content/drive
Repo dir: /content/sharp-har
Checkpoint root: /content/drive/MyDrive/sharp_har_runs


In [2]:



# Diagnostic (NOT part of the frozen §7 protocol): is the domain linearly
# readable from the encoder features at all, with or without the GRL?
#
# Motivation: the §7 ar_set probe on the p2_lab val split is uninformative
# (9 traces, AR-3 absent, AR-1 == AR-2 in every metadata attribute), and C2's
# arset_train_acc sat at the train majority baseline (0.2969) from epoch 2 on,
# with lambda still ramping — so it is unclear whether the GRL suppressed the
# adversary or the adversary never had anything to learn.
#
# This runs the frozen §5.3 recipe (probe.linear_probe, untouched) on an inner
# TRACE-DISJOINT stratified split of the 81 train traces, over several targets.
# Train features only: no val selection, no test contact (§0.7).
#
# Declared confound: the encoder was trained on ALL 81 train traces, so the
# absolute numbers are inflated by memorization. The C1-vs-C2 CONTRAST stays
# fair - both encoders saw exactly the same traces.

from __future__ import annotations

from pathlib import Path

import numpy as np

from sharp_har.harness import fuse_windows
from sharp_har.inventory import AR_SET_METADATA
from sharp_har.probe import linear_probe, majority_baseline

RUNS = CKPT_ROOT
SEED = 42          # §0 rule 5
EVAL_FRAC = 1 / 3
# "y" is the POSITIVE CONTROL, not a result: activity is known to be readable
# from these features (C1-lin scored 0.8835 on val). If y collapses on the
# inner split too, this diagnostic is broken and every other row is void.
TARGETS = ("y", "ar_set", "ambiente", "direct_path", "persona", "monitor")


def build_targets(arset_name: np.ndarray, arset_int: np.ndarray, arset_labels: np.ndarray) -> dict:
    """ar_set (the adversary's actual target) + the metadata attributes it
    conflates. The ar_set label space comes from the cache's own arset_labels
    (the train-domain index order the integers refer to); the derived spaces
    are built from the AR-sets OBSERVED here, so no class is allocated for
    AR-7 (test, absent). Identical across runs: same frozen train trace list."""
    names = np.array([str(a) for a in arset_name])
    ar_labels = [str(a) for a in arset_labels]
    assert int(arset_int.max()) < len(ar_labels), "ar_set index outside arset_labels"
    assert set(names) <= set(ar_labels), "arset_name outside the train-domain label set"
    out = {"ar_set": (arset_int.astype(np.int64), ar_labels)}
    for attr in ("ambiente", "direct_path", "persona", "monitor"):
        vals = sorted({AR_SET_METADATA[n][attr] for n in set(names)})
        out[attr] = (
            np.array([vals.index(AR_SET_METADATA[n][attr]) for n in names], dtype=np.int64),
            vals,
        )
    return out


def inner_split(trace_id: np.ndarray, arset_int: np.ndarray, seed: int = SEED):
    """Trace-disjoint split stratified by AR-set (§0 rule 2: never by window).
    Deterministic: sorted trace order, then one seeded permutation per AR-set."""
    trace_to_ar: dict[str, int] = {}
    for t, a in zip(trace_id, arset_int):
        trace_to_ar.setdefault(str(t), int(a))

    by_ar: dict[int, list[str]] = {}
    for t, a in trace_to_ar.items():
        by_ar.setdefault(a, []).append(t)

    rng = np.random.default_rng(seed)
    fit, evl = [], []
    for a in sorted(by_ar):
        ts = np.array(sorted(by_ar[a]), dtype=object)
        ts = ts[rng.permutation(len(ts))]
        n_ev = max(1, int(round(len(ts) * EVAL_FRAC)))
        evl += list(ts[:n_ev])
        fit += list(ts[n_ev:])

    fit_s, ev_s = set(fit), set(evl)
    assert not (fit_s & ev_s), "inner split is not trace-disjoint (§0 rule 2)"
    assert fit_s and ev_s
    return fit_s, ev_s


def fused_accuracy(x, y, tid, ws, weight, bias) -> tuple[float, float]:
    """Antenna-fused accuracy of the probe head + the majority baseline on the
    SAME fused units (§7 reads accuracy against the majority class, not 1/n)."""
    logits = x @ weight.T + bias
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = (np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)).astype(np.float32)
    fused = fuse_windows(probs, y, tid.astype(object), ws)
    acc = float((fused["y_pred"] == fused["y_true"]).mean())
    return acc, majority_baseline(fused["y_true"])


def diagnose(run_dir: Path, label: str) -> list[dict]:
    path = run_dir / "features_best_train.npz"
    if not path.exists():
        print(f"[{label}] SKIP - {path} not found (cached by probe_encoder next to best.ckpt)")
        return []

    d = np.load(path, allow_pickle=False)
    x, tid, ws = d["features"], d["trace_id"], d["window_start"]
    targets = build_targets(d["arset_name"], d["ar_set"], d["arset_labels"])
    targets["y"] = (d["y"].astype(np.int64), [str(l) for l in d["labels"]])
    fit_s, ev_s = inner_split(tid, d["ar_set"])

    m_fit = np.array([str(t) in fit_s for t in tid])
    m_ev = np.array([str(t) in ev_s for t in tid])

    print(f"\n[{label}] {path.name}: {x.shape[0]} samples, d={x.shape[1]}")
    print(f"[{label}] inner split: {len(fit_s)} fit traces / {len(ev_s)} eval traces "
          f"({m_fit.sum()} / {m_ev.sum()} samples)")

    rows = []
    for target in TARGETS:
        y, vals = targets[target]
        n_cls = len(vals)
        present = sorted(set(y[m_ev].tolist()))
        fit_d = {"features": x[m_fit], target: y[m_fit], "trace_id": tid[m_fit], "window_start": ws[m_fit]}
        ev_d = {"features": x[m_ev], target: y[m_ev], "trace_id": tid[m_ev], "window_start": ws[m_ev]}

        res = linear_probe(fit_d, ev_d, target=target, n_classes=n_cls, seed=SEED)
        acc, base = fused_accuracy(
            x[m_ev], y[m_ev], tid[m_ev], ws[m_ev], res["weight"], res["bias"]
        )
        rows.append({
            "run": label, "target": target, "n_classes": n_cls,
            "eval_accuracy": acc, "majority_baseline": base, "delta": acc - base,
            "macro_f1": res["best_val_macro_f1"], "best_epoch": res["best_epoch"],
            "classes_present_in_eval": len(present), "labels": vals,
        })
        tag = "  <- CONTROL" if target == "y" else ""
        print(f"[{label}] {target:12s} ({n_cls} cls): acc {acc:.3f} vs baseline {base:.3f} "
              f"(delta {acc - base:+.3f}), macro-F1 {res['best_val_macro_f1']:.3f}{tag}")
    return rows


if __name__ == "__main__":
    rows = []
    for label, sub in (("C1", "C1"), ("C2", "C2")):
        rows += diagnose(RUNS / sub, label)

    print("\n" + "=" * 78)
    print(f"{'run':4s} {'target':12s} {'acc':>7s} {'baseline':>9s} {'delta':>8s} {'macroF1':>8s}")
    print("-" * 78)
    for r in rows:
        print(f"{r['run']:4s} {r['target']:12s} {r['eval_accuracy']:7.3f} "
              f"{r['majority_baseline']:9.3f} {r['delta']:+8.3f} {r['macro_f1']:8.3f}")
    print("=" * 78)
    print("\nREAD THE CONTROL FIRST: y must land near the C1-lin val score (~0.88)")
    print("with a large positive delta. If it does not, the inner split is broken")
    print("and no other row means anything.")
    print("\nOnly if the control passes: delta ~ 0 on the domain targets => the")
    print("domain is NOT linearly readable from these features, so the GRL had")
    print("nothing to remove (and C2 paid for nothing). delta >> 0 for C1 but ~ 0")
    print("for C2 => the GRL works and the val probe was merely underpowered.")
    print("ar_set is what the adversary optimizes; ambiente is what §9 claims.")


[C1] features_best_train.npz: 53400 samples, d=256
[C1] inner split: 55 fit traces / 26 eval traces (35584 / 17816 samples)


2026-07-17 07:41:12,546 [INFO] sharp_har.probe: linear probe [y]: best val macro-F1 0.9995 at epoch 5/10


[C1] y            (8 cls): acc 1.000 vs baseline 0.197 (delta +0.803), macro-F1 1.000  <- CONTROL


2026-07-17 07:41:16,001 [INFO] sharp_har.probe: linear probe [ar_set]: best val macro-F1 0.0662 at epoch 4/9


[C1] ar_set       (6 cls): acc 0.196 vs baseline 0.286 (delta -0.090), macro-F1 0.066


...

[C1] direct_path  (2 cls): acc 0.633 vs baseline 0.731 (delta -0.098), macro-F1 0.446


2026-07-17 07:41:26,117 [INFO] sharp_har.probe: linear probe [persona]: best val macro-F1 0.4500 at epoch 1/6


[C1] persona      (2 cls): acc 0.818 vs baseline 0.818 (delta +0.000), macro-F1 0.450


2026-07-17 07:41:32,669 [INFO] sharp_har.probe: linear probe [monitor]: best val macro-F1 0.2806 at epoch 13/18


[C1] monitor      (3 cls): acc 0.499 vs baseline 0.584 (delta -0.086), macro-F1 0.281
[C2] SKIP - /content/drive/MyDrive/sharp_har_runs/C2/features_best_train.npz not found (cached by probe_encoder next to best.ckpt)

run  target           acc  baseline    delta  macroF1
------------------------------------------------------------------------------
C1   y              1.000     0.197   +0.803    1.000
C1   ar_set         0.196     0.286   -0.090    0.066
C1   ambiente       0.854     0.854   +0.000    0.461
C1   direct_path    0.633     0.731   -0.098    0.446
C1   persona        0.818     0.818   +0.000    0.450
C1   monitor        0.499     0.584   -0.086    0.281

READ THE CONTROL FIRST: y must land near the C1-lin val score (~0.88)
with a large positive delta. If it does not, the inner split is broken
and no other row means anything.

Only if the control passes: delta ~ 0 on the domain targets => the
domain is NOT linearly readable from these features, so the GRL had
nothing to rem